# 🤖 Code Review Agent — Explication complète

Ce notebook explique **comment fonctionne** l'agent de code review que nous avons construit,
étape par étape, du concept jusqu'au déploiement.

---

## Qu'est-ce qu'un AI Agent ?

Un LLM classique répond à une question en **une seule fois** — il génère du texte, c'est tout.

Un **agent** peut :
- **Décider** quoi faire
- **Appeler des outils** (fonctions Python, APIs, fichiers...)
- **Observer** les résultats
- **Recommencer** jusqu'à atteindre l'objectif

```
LLM simple :  question → réponse

Agent :       question → [réfléchit] → [appelle outil] → [observe] → [réfléchit] → réponse finale
```

Ce pattern s'appelle **ReAct** : **Re**asoning + **Act**ing.

---

## Architecture du projet

```
fichier Python à analyser
         ↓
      review.py          ← point d'entrée
         ↓
   code_reviewer/
   ├── agent.py          ← LangGraph orchestre tout
   └── tools.py          ← 4 fonctions d'analyse spécialisées
         ↓
   Rapport Markdown
         ↓
   Commentaire PR GitHub (via GitHub Action)
```

### Stack
| Composant | Rôle |
|-----------|------|
| **LangGraph** | Orchestre les appels de l'agent |
| **LangChain** | Interface avec les LLMs |
| **Ollama** (local) | LLM en développement, sans coût |
| **Groq** (cloud) | LLM en production sur GitHub Actions |
| **GitHub Actions** | Exécute l'agent sur chaque Pull Request |

---

## Étape 1 — Choisir le LLM selon l'environnement

L'agent doit fonctionner dans deux contextes :
- **Local** : Ollama tourne sur ta machine (gratuit, privé)
- **GitHub Actions** : pas d'Ollama possible → on utilise Groq (API cloud gratuite)

La solution : on vérifie si la variable `GROQ_API_KEY` est définie.
Si oui → Groq. Sinon → Ollama.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()  # charge les variables depuis .env

def get_llm():
    if os.getenv("GROQ_API_KEY"):
        from langchain_groq import ChatGroq
        print("→ Mode PRODUCTION : Groq (llama-3.3-70b)")
        return ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
    else:
        from langchain_ollama import ChatOllama
        print("→ Mode LOCAL : Ollama (qwen2.5:3b)")
        return ChatOllama(model="qwen2.5:3b")

llm = get_llm()

→ Mode PRODUCTION : Groq (llama-3.3-70b)


---

## Étape 2 — Les Tools (outils)

Un **tool** est une simple fonction Python décorée avec `@tool`.
Le **docstring** est crucial : c'est ce que le LLM lit pour décider **quand** appeler cet outil.

On a 4 outils, chacun spécialisé dans un aspect de la review :

| Tool | Ce qu'il analyse |
|------|------------------|
| `analyser_style` | PEP8, nommage, complexité |
| `detecter_bugs` | Erreurs logiques, edge cases |
| `verifier_securite` | Credentials, injections |
| `suggerer_refactoring` | Améliorations de structure |

In [2]:
from langchain_core.tools import tool

def _llm_analyze(prompt: str) -> str:
    """Fonction interne : envoie un prompt au LLM et retourne la réponse."""
    return get_llm().invoke(prompt).content


@tool
def analyser_style(code: str) -> str:
    """Check PEP8 compliance, naming conventions, function length, and complexity."""
    return _llm_analyze(
        "You are a Python style reviewer. List all style issues (PEP8 violations, "
        "poor naming, functions over 20 lines, high complexity). "
        "Be specific. If no issues, say so.\n\n"
        f"Code:\n```python\n{code}\n```"
    )

@tool
def detecter_bugs(code: str) -> str:
    """Find potential bugs, logic errors, unhandled edge cases, and broad exceptions."""
    return _llm_analyze(
        "You are a Python bug hunter. Find potential bugs, logic errors, "
        "and unhandled edge cases. Be specific. If no issues, say so.\n\n"
        f"Code:\n```python\n{code}\n```"
    )

@tool
def verifier_securite(code: str) -> str:
    """Find security vulnerabilities: hardcoded credentials, injections, exposed data."""
    return _llm_analyze(
        "You are a security auditor. Find security issues: hardcoded credentials, "
        "SQL injection risks, use of eval/exec. Be specific. If no issues, say so.\n\n"
        f"Code:\n```python\n{code}\n```"
    )

@tool
def suggerer_refactoring(code: str) -> str:
    """Suggest the top refactoring improvements: extract functions, reduce duplication."""
    return _llm_analyze(
        "You are a Python architect. Suggest the top 3 refactoring improvements. "
        "Be concise and actionable.\n\n"
        f"Code:\n```python\n{code}\n```"
    )

tools = [analyser_style, detecter_bugs, verifier_securite, suggerer_refactoring]
print(f"{len(tools)} tools définis : {[t.name for t in tools]}")

4 tools définis : ['analyser_style', 'detecter_bugs', 'verifier_securite', 'suggerer_refactoring']


### Test d'un tool en isolation

Avant de tout assembler, on peut tester un tool seul pour vérifier qu'il fonctionne.

In [3]:
code_exemple = '''
import os

password = "admin123"  # hardcoded !

def f(x, y, z, a, b):
    try:
        result = x / y
        return result
    except:
        return 0  # exception trop large
'''

# Test direct du tool — l'agent ne l'appelle pas encore, c'est nous qui le faisons
resultat = verifier_securite.invoke({"code": code_exemple})
print(resultat)

→ Mode PRODUCTION : Groq (llama-3.3-70b)
**Security Issues Found:**

1. **Hardcoded Credentials**: The password "admin123" is hardcoded in the script. This is a significant security risk as it can be easily accessed by anyone with read access to the code. Hardcoded credentials should be avoided and instead, consider using environment variables or a secure secrets management system.

2. **Broad Exception Handling**: The `except` block in the `f` function is too broad, catching all exceptions. This can mask potential security issues, such as division by zero or type errors, making it difficult to diagnose problems. It's better to catch specific exceptions that can occur during the execution of the code.

**No Issues Found:**

* No SQL injection risks are present in this code, as there are no database interactions.
* No use of `eval` or `exec` functions is present in this code, which can pose a significant security risk if used with untrusted input.

**Recommendations:**

* Remove the har

---

## Étape 3 — L'Agent LangGraph

### Pourquoi LangGraph ?

LangGraph modélise l'agent comme un **graphe d'état** :

```
         ┌─────────────────────────────┐
START ──→│          agent              │
         │  (décide quoi faire)        │
         └──────────────┬──────────────┘
                        │  tool_call
                        ↓
         ┌─────────────────────────────┐
         │          tools              │
         │  (exécute la fonction)      │
         └──────────────┬──────────────┘
                        │  résultat
                        ↓
                    [agent]  ← recommence ou termine
                        │  (si fini)
                        ↓
                       END
```

L'agent tourne en **boucle** jusqu'à décider qu'il a assez d'informations pour répondre.

### Le System Prompt — comment on guide l'agent

Le `prompt` (system prompt) est une **instruction permanente** qui dit à l'agent
comment se comporter. Sans lui, l'agent pourrait oublier d'appeler certains tools.

In [4]:
from langgraph.prebuilt import create_react_agent  # noqa

SYSTEM_PROMPT = """You are a senior Python code reviewer.

When given code to review, you MUST:
1. Call analyser_style with the full code
2. Call detecter_bugs with the full code
3. Call verifier_securite with the full code
4. Call suggerer_refactoring with the full code
5. After ALL 4 tools have returned, write the final Markdown report.

Final report format:
## 🔴 Bugs
## 🟡 Security
## 🔵 Style
## ✅ Refactoring Suggestions
## Summary

Call all tools before writing the report. Do not skip any tool."""

# create_react_agent crée le graphe LangGraph avec :
# - llm : le cerveau qui décide
# - tools : les fonctions qu'il peut appeler
# - prompt : les instructions permanentes
agent = create_react_agent(llm, tools=tools, prompt=SYSTEM_PROMPT)

print("Agent créé.")
print(f"LLM : {llm.__class__.__name__}")
print(f"Tools disponibles : {[t.name for t in tools]}")

Agent créé.
LLM : ChatGroq
Tools disponibles : ['analyser_style', 'detecter_bugs', 'verifier_securite', 'suggerer_refactoring']


C:\Users\amine\AppData\Local\Temp\ipykernel_23172\1213681092.py:25: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools=tools, prompt=SYSTEM_PROMPT)


---

## Étape 4 — Exécution avec streaming

On utilise `.stream()` au lieu de `.invoke()` pour voir **chaque étape** de l'agent en temps réel.

Chaque `step` retourné est un dict avec :
- clé `"agent"` → l'agent a pris une décision (tool call ou réponse finale)
- clé `"tools"` → un tool a été exécuté et a retourné un résultat

In [ ]:
code_a_reviewer = '''
import os

API_KEY = "sk-1234abcd"  # hardcoded secret

def process(data, flag, mode, option, extra):
    try:
        result = eval(data)  # dangereux !
        if flag == True:
            return result
        return None
    except:
        pass  # silently ignores all errors
'''

print("=" * 60)
print("DÉMARRAGE DE LA REVIEW")
print("=" * 60)

final_report = ""

for step in agent.stream(
    {"messages": [("user", f"Review this Python code:\n\n```python\n{code_a_reviewer}\n```")]},
    stream_mode="updates"
):
    for node, data in step.items():
        for msg in data.get("messages", []):
            # L'agent appelle un tool
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"\n[AGENT → TOOL] {tc['name']}(...)")
            # Un tool retourne un résultat
            elif node == "tools" and hasattr(msg, "content"):
                print(f"[TOOL → AGENT] résultat reçu ({len(msg.content)} chars)")
            # L'agent génère la réponse finale
            elif node == "agent" and hasattr(msg, "content") and msg.content:
                final_report = msg.content

print("\n" + "=" * 60)
print("RAPPORT FINAL")
print("=" * 60)
print(final_report)

---

## Étape 5 — Le point d'entrée CLI (`review.py`)

`review.py` permet d'utiliser l'agent depuis le terminal ou depuis la GitHub Action :

```bash
python review.py mon_fichier.py
```

Il fait 3 choses simples :
1. Lit le fichier passé en argument
2. Lance l'agent sur le code
3. Affiche le rapport + le sauvegarde en `.md`

```python
# review.py (simplifié pour la lecture)
import sys
from pathlib import Path
from code_reviewer.agent import review_code

file_path = Path(sys.argv[1])
code = file_path.read_text(encoding="utf-8")
report = review_code(code)         # ← appel à l'agent
print(report)
```

---

## Étape 6 — La GitHub Action

Le fichier `.github/workflows/code-review.yml` automatise tout sur GitHub.

### Déclencheur
```yaml
on:
  pull_request:
    types: [opened, synchronize, reopened]
```
→ L'action se lance **à chaque Pull Request** ouverte ou mise à jour.

### Déroulement
```
1. Checkout du code de la PR
2. Installation des dépendances (pip install -r requirements.txt)
3. Récupération des fichiers .py modifiés (git diff)
4. Pour chaque fichier → python review.py <fichier>
5. Le rapport est posté comme commentaire sur la PR via l'API GitHub
```

### La clé Groq dans GitHub Secrets
L'action ne peut pas utiliser Ollama (pas de GPU sur les runners GitHub).
On stocke `GROQ_API_KEY` dans **Settings → Secrets** du repo.
Elle est injectée via :
```yaml
env:
  GROQ_API_KEY: ${{ secrets.GROQ_API_KEY }}
```
Notre `get_llm()` la détecte automatiquement et bascule sur Groq.

---

## Résumé — Ce que tu as construit

```
┌─────────────────────────────────────────────────────────┐
│                   CODE REVIEW AGENT                     │
│                                                         │
│  Input  : fichier Python                                │
│                                                         │
│  Agent  : LangGraph ReAct loop                          │
│    ├── Tool 1 : analyser_style    (LLM spécialisé)      │
│    ├── Tool 2 : detecter_bugs     (LLM spécialisé)      │
│    ├── Tool 3 : verifier_securite (LLM spécialisé)      │
│    └── Tool 4 : suggerer_refactoring (LLM spécialisé)   │
│                                                         │
│  Output : rapport Markdown structuré                    │
│                                                         │
│  Deploy : GitHub Action → commentaire PR automatique    │
└─────────────────────────────────────────────────────────┘
```

### Ce que tu peux dire en entretien

> *"J'ai construit un agent IA avec LangGraph qui analyse du code Python en utilisant 4 outils
> spécialisés en parallèle — style, bugs, sécurité, refactoring. L'agent orchestre les appels
> via un graphe d'état ReAct et synthétise les résultats en un rapport structuré.
> Il est déployé comme GitHub Action : à chaque Pull Request, il poste automatiquement
> un commentaire de review. En local j'utilise Ollama (zéro coût), en prod Groq API."
> *

**Concepts démontrés :** AI Agents · LangGraph · ReAct Pattern · Tool Use ·
Prompt Engineering · LLM orchestration · CI/CD · GitHub Actions